In [1]:
import os
import sys
import glob
import torch
import librosa
import warnings
import numpy as np
import torch.nn as nn
from time import sleep
from torch import optim
import torch.nn.functional as F
from torch.autograd import Variable
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
warnings.filterwarnings('ignore')

In [2]:
# Caution: Do not change the default parameters
def get_features(filepath, sr=8000, n_mfcc=30, n_mels=128, frames = 15):
    # The following function contains code to produce features of the audio sample.
    y, sr = librosa.load(filepath, sr=sr)
    D = np.abs(librosa.stft(y))**2
    S = librosa.feature.melspectrogram(S=D)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    log_S = librosa.power_to_db(S,ref=np.max)
    features = librosa.feature.mfcc(S=log_S, n_mfcc=n_mfcc)
    if features.shape[1] < frames :
        features = np.hstack((features, np.zeros((n_mfcc, frames - features.shape[1]))))
    elif features.shape[1] > frames:
        features = features[:, :frames]

    # Find 1st order delta_mfcc
    delta1_mfcc = librosa.feature.delta(features, order=1)

    # Find 2nd order delta_mfcc
    delta2_mfcc = librosa.feature.delta(features, order=2)

    # Stacking delta_mfcc features in sequence horizontally (column wise)
    features = np.hstack((delta1_mfcc.flatten(), delta2_mfcc.flatten()))

    # Increase the dimension by inserting an axis along second dimension
    features = features.flatten()[:,np.newaxis]

    # Convert the numpy.ndarray to a Tensor object
    features = Variable(torch.from_numpy(features)).float()
    return features

In [3]:
def load_data(folder_path):
    features = []
    labels = []

    for file in os.listdir(folder_path):
        if file.endswith('.wav'):
            filepath = os.path.join(folder_path, file)
            feat = get_features(filepath)
            features.append(feat.numpy().flatten())
            label = int(file.split('_')[0])
            labels.append(label)
    features = np.array(features)
    labels = np.array(labels)
    return features, labels

In [4]:
features, labels = load_data("C:/Users/satyasrp/personal/projects/aiml/audio/studio_data/studio_data")

In [5]:
features.shape

(3979, 900)

In [6]:
labels.shape

(3979,)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42, stratify=labels)

In [8]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [9]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
X_train_tensor[0]
y_train_tensor[0]
train_dataset[0]


(tensor([-3.0656e+01, -3.0656e+01, -3.0656e+01, -3.0656e+01, -3.0656e+01,
         -5.3154e+01, -5.3665e+01, -4.4649e+01, -3.7908e+01, -4.2372e+01,
         -4.5441e+01, -4.5441e+01, -4.5441e+01, -4.5441e+01, -4.5441e+01,
          1.4897e+01,  1.4897e+01,  1.4897e+01,  1.4897e+01,  1.4897e+01,
          1.6861e+01,  1.6803e+01,  1.5173e+01,  1.2173e+01,  9.3953e+00,
          1.1256e+00,  1.1256e+00,  1.1256e+00,  1.1256e+00,  1.1256e+00,
         -4.2077e+00, -4.2077e+00, -4.2077e+00, -4.2077e+00, -4.2077e+00,
         -3.2129e+00, -1.3843e+00,  2.0006e+00,  7.8690e+00,  1.2250e+01,
          1.1993e+01,  1.1993e+01,  1.1993e+01,  1.1993e+01,  1.1993e+01,
          8.4624e-01,  8.4624e-01,  8.4624e-01,  8.4624e-01,  8.4624e-01,
          7.8432e-01,  1.5245e+00,  2.3754e+00,  1.8977e+00, -2.0993e-01,
         -1.3689e+00, -1.3689e+00, -1.3689e+00, -1.3689e+00, -1.3689e+00,
         -5.4022e-01, -5.4022e-01, -5.4022e-01, -5.4022e-01, -5.4022e-01,
          1.5407e+00,  4.0673e+00,  5.

In [10]:
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [11]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [30]:
class Net(nn.Module):
    def __init__(self,num_classes=6):
        super(Net, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=900, out_channels=400, kernel_size=1)
        self.bn1 = nn.BatchNorm1d(400)
        self.relu1 = nn.ReLU()
        self.maxpool1 = nn.MaxPool1d(1)
        self.dropout1 = nn.Dropout(p=0.25)
        self.conv2 = nn.Conv1d(in_channels=400, out_channels=200, kernel_size=1)
        self.bn2 = nn.BatchNorm1d(200)
        self.relu2 = nn.ReLU()
        self.maxpool2 = nn.MaxPool1d(1)
        self.dropout2 = nn.Dropout(p=0.25)
        self.conv3 = nn.Conv1d(in_channels=200, out_channels=100, kernel_size=1)
        self.bn3 = nn.BatchNorm1d(100)
        self.relu3 = nn.ReLU()
        self.maxpool3 = nn.MaxPool1d(1)
        self.dropout3 = nn.Dropout(p=0.25)
        self.fc1 = nn.Linear(100, num_classes)
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = x.view(x.size(0), 900, -1)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.maxpool1(out)
        out = self.dropout1(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu2(out)
        out = self.maxpool2(out)
        out = self.dropout2(out)
        out = self.conv3(out)
        out = self.bn3(out)
        out = self.relu3(out)
        out = self.maxpool3(out)
        out = self.dropout3(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.log_softmax(out)

        return out

class RegularizedMLP(nn.Module):
    def __init__(self, input_size=900, num_classes=6, dropout_rate=0.6):
        super(RegularizedMLP, self).__init__()
        
        self.fc1 = nn.Linear(input_size, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.dropout1 = nn.Dropout(dropout_rate)
        
        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.dropout3 = nn.Dropout(dropout_rate)
        
        self.fc4 = nn.Linear(128, 64)
        self.bn4 = nn.BatchNorm1d(64)
        self.dropout4 = nn.Dropout(dropout_rate)
        
        self.fc5 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten
        
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        
        x = F.relu(self.bn3(self.fc3(x)))
        x = self.dropout3(x)
        
        x = F.relu(self.bn4(self.fc4(x)))
        x = self.dropout4(x)
        
        x = self.fc5(x)
        
        return F.log_softmax(x, dim=1)

In [32]:
def train_improved_model(model_class, model_name, train_loader, test_loader):
    model = model_class()
    model = model.to(device)
    print(f"\nTraining {model_name}...")
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion = nn.NLLLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    best_val_acc = 0
    train_losses = []
    val_accs = []
    
    for epoch in range(10):
        model.train()
        running_loss = 0
        correct = 0
        total = 0
        
        for batch_features, batch_labels in train_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0
        
        with torch.no_grad():
            for val_features, val_labels in test_loader:
                val_features, val_labels = val_features.to(device), val_labels.to(device)
                val_outputs = model(val_features)
                val_loss += criterion(val_outputs, val_labels).item()
                
                _, predicted = torch.max(val_outputs, 1)
                val_total += val_labels.size(0)
                val_correct += (predicted == val_labels).sum().item()
        
        val_acc = 100 * val_correct / val_total
        epoch_loss = running_loss / len(train_loader)
        val_loss /= len(test_loader)
        
        train_losses.append(epoch_loss)
        val_accs.append(val_acc)
        
        scheduler.step(val_loss)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"best_{model_name.lower().replace(' ', '_')}.pth")
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%  Loss={epoch_loss:.4f}, Val Acc={val_acc:.2f}%")
    
    print(f"{model_name} - Best Validation Accuracy: {best_val_acc:.2f}%")
    return model, best_val_acc, train_losses, val_accs

# Test all models
models_to_test = [
    (Net, "CNN"),
    (RegularizedMLP, "Regularized MLP")
]

results = {}
for model_class, model_name in models_to_test:
    model, best_acc, losses, accs = train_improved_model(
        model_class, model_name, train_loader, test_loader
    )
    results[model_name] = {
        'model': model,
        'accuracy': best_acc,
        'losses': losses,
        'accuracies': accs
    }

# Compare results
print("\n=== Model Comparison ===")
for name, result in results.items():
    print(f"{name}: {result['accuracy']:.2f}%")

best_model_name = max(results.keys(), key=lambda k: results[k]['accuracy'])
print(f"\nBest Model: {best_model_name} with {results[best_model_name]['accuracy']:.2f}% accuracy")


Training CNN...
Epoch 10: Train Loss: 0.4072, Train Acc: 84.45%  Loss=0.4072, Val Acc=84.67%
CNN - Best Validation Accuracy: 84.67%

Training Regularized MLP...
Epoch 10: Train Loss: 1.0722, Train Acc: 58.72%  Loss=1.0722, Val Acc=73.49%
Regularized MLP - Best Validation Accuracy: 73.49%

=== Model Comparison ===
CNN: 84.67%
Regularized MLP: 73.49%

Best Model: CNN with 84.67% accuracy
